In [1]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
nn_architecture = [
    {"input_dim": 2, "output_dim": 4, "activation": "relu"},
    {"input_dim": 4, "output_dim": 6, "activation": "relu"},
    {"input_dim": 6, "output_dim": 6, "activation": "relu"},
    {"input_dim": 6, "output_dim": 4, "activation": "relu"},
    {"input_dim": 4, "output_dim": 1, "activation": "sigmoid"},
]

In [3]:
def init_layers(nn_architecture, seed=99):
    np.random.seed(seed)
    params_values = {}
    for idx, layer in enumerate(nn_architecture):
        layer_idx = idx + 1
        params_values["W" + str(layer_idx)] = np.random.randn(
            layer["output_dim"], layer["input_dim"]) * 0.1
        params_values["b" + str(layer_idx)] = np.zeros((layer["output_dim"], 1))
    return params_values

In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(x))

def relu(x):
    return x if x > 0 else 0

def sigmoid_backward(dA, x):
    sig = sigmoid(x)
    return dA * sig * (1 - sig)

def relu_backward(dA, x):
    dx = np.array(dA, copy=True)
    dx[x <= 0] = 0
    return dx

In [ ]:
def conver_prob_into_class(probs):
    """Converts probabilities intro binary classes (0 or 1)."""
    return (probs > 0.5).astype(int)

In [ ]:
def single_layer_forward_propagation(A_prev, W_curr, b_curr, activation="relu"):
    Z_curr = np.dot(W_curr, A_prev) + b_curr
    if activation == "relu":
        activation_func = relu
    elif activation == "sigmoid":
        activaton_func = sigmoid
    else:
        raise Exception("Non-supported activation function")
    return activation_func(Z_curr), Z_curr

In [ ]:
def full_forward_propagation(X, params_values, nn_architecture):
    memory = {}
    A_curr = X
    for idx, layer in enumerate(nn_architecture):
        layer_idx = idx + 1
        A_prev = A_curr
        A_curr, Z_curr = single_layer_forward_propagation(
            A_prev, params_values["W" + str(layer_idx)],
            params_values["b" + str(layer_idx)],
            layer["activation"]
        )
        memory["A" + str(idx)] = A_prev
        memory["Z" + str(layer_idx)] = Z_curr
    return A_curr, memory
    

In [ ]:
# Cross Entropy
def get_cost_value(Y_hat, Y):
    m = Y_hat.shape(1)
    cost = -1 / m * (np.dot(Y, np.log(Y_hat).T) + np.dot(1 - Y, np.log(1 - Y_hat).T))
    return np.squeeze(cost)

In [ ]:
def get_accuracy_value(Y_hat, Y):
    Y_hat_class = conver_prob_into_class(Y_hat)
    return (Y_hat_class == Y).mean()

In [ ]:
def single_layer_backward_propagation(dA_curr, W_curr, b_curr, Z_curr, A_prev, activation="relu"):
    m = A_prev.shape[1]
    if activation == "relu":
        backward_activation_func = relu_backward
    elif activation == "sigmoid":
        backward_activation_func = sigmoid_backward
    else:
        raise Exception("Non-supported activation function")
    
    dZ_curr = backward_activation_func(dA_curr, Z_curr)
    dW_curr = np.dot(dZ_curr, A_prev.T) / m
    db_curr = np.sum(dZ_curr, axis=1, keepdims=True)
    dA_prev = np.dot(W_curr.T, dZ_curr)
    return dA_prev, dW_curr, db_curr

In [ ]:
def full_backward_propagation(Y_hat, Y, memory, params_values, nn_architecture):
    grads_values = {}
    Y = Y.reshape(Y_hat.shape)
    dA_prev = - (np.divide(Y, Y_hat) - np.divide(1 - Y, 1 - Y_hat))

    for layer_idx_prev, layer in reversed(list(enumerate(nn_architecture))):
        layer_idx_curr = layer_idx_prev + 1
        dA_prev, dW_curr, db_curr = single_layer_backward_propagation(
            dA_prev,
            params_values["W" + str(layer_idx_curr)],
            params_values["b" + str(layer_idx_curr)],
            memory["Z" + str(layer_idx_curr)],
            memory["A" + str(layer_idx_prev)],
            layer["activation"]
        )
        grads_values["dW" + str(layer_idx_curr)] = dW_curr
        grads_values["db" + str(layer_idx_curr)] = db_curr
    return grads_values